# NB_DOC_01 · Extraer la estructura y una muestra del modelo semántico

**Paso 1 del framework. Solo lectura**: no modifica el modelo.

Lee tablas, columnas, medidas y relaciones con TOM (vía `semantic-link-labs`), calcula filas, nulos y cardinalidad con DAX, toma una muestra por tabla (enmascarando columnas sensibles) y deja en el lakehouse por defecto:

| Archivo | Para qué |
|---|---|
| `Files/doc_modelos/<modelo>/modelo.json` | Todo lo extraído. Es lo que lee Claude en el paso 2 |
| `Files/doc_modelos/<modelo>/descripciones.csv` | Una fila por tabla, columna y medida, con la descripción actual y la columna `descripcion_nueva` vacía para completar |

Requisitos: un lakehouse por defecto adjunto al notebook, el endpoint XMLA habilitado en la capacidad (lectura alcanza para este notebook) y permiso de **Build/Write** sobre el modelo.

In [ ]:
# Parámetros (celda marcada como "parameters": se pueden pasar desde un pipeline)
MODELO = "<nombre-del-modelo-semantico>"
WORKSPACE = "<nombre-del-workspace>"   # None = el workspace del notebook
FILAS_MUESTRA = 5                      # filas de ejemplo por tabla
CALCULAR_NULOS = True                  # COUNTBLANK por columna (en modelos muy grandes puede tardar)
CARPETA_SALIDA = "/lakehouse/default/Files/doc_modelos"

# Nombres de columna que se tratan como sensibles: no se guardan ejemplos.
# Ajustalo a tu negocio. Coincide por substring, sin distinguir mayúsculas.
PATRONES_SENSIBLES = [
    "email", "mail", "dni", "cuit", "cuil", "documento", "telefono", "phone", "celular",
    "direccion", "address", "domicilio", "nombre", "apellido", "name", "cbu", "iban",
    "tarjeta", "card", "password", "pass", "token", "nacimiento", "birth", "salario", "sueldo",
]
# Fuente única compartida con NB_PRF_01: si existe Files/comun/patrones_sensibles.json
# (copia de Comun/patrones_sensibles.json del repo BIA), reemplaza la lista de arriba.
RUTA_PATRONES_SENSIBLES = "/lakehouse/default/Files/comun/patrones_sensibles.json"

In [ ]:
# Versión fija, la misma que Performance_Modelo_Semantico (lee el modelo.json de este notebook).
%pip install -q semantic-link-labs==0.17.1

In [ ]:
import json, os, re
import json as _json, os as _os
if _os.path.exists(RUTA_PATRONES_SENSIBLES):
    with open(RUTA_PATRONES_SENSIBLES, encoding="utf-8") as _f:
        PATRONES_SENSIBLES = [p.lower() for p in _json.load(_f)["patrones"]]
    print(f"PATRONES_SENSIBLES: {len(PATRONES_SENSIBLES)} patrones de {RUTA_PATRONES_SENSIBLES}")
from datetime import datetime, timezone

import pandas as pd
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model


def es_sensible(nombre_columna: str) -> bool:
    n = nombre_columna.lower()
    return any(p in n for p in PATRONES_SENSIBLES)


def dax_tabla(nombre: str) -> str:
    """Referencia DAX segura a una tabla: 'Mi tabla' (escapa comillas simples)."""
    return "'" + nombre.replace("'", "''") + "'"


def dax_columna(tabla: str, columna: str) -> str:
    return dax_tabla(tabla) + "[" + columna.replace("]", "]]") + "]"

## 1. Metadatos con TOM (solo lectura)

In [ ]:
tablas, medidas, relaciones = [], [], []

with connect_semantic_model(dataset=MODELO, workspace=WORKSPACE, readonly=True) as tom:
    for t in tom.model.Tables:
        modos = sorted({str(p.Mode) for p in t.Partitions})
        # De dónde lee cada partición: Direct Lake (esquema y tabla del lakehouse) o expresión M.
        # Lo usa Catalogo_Linaje para unir el modelo con gold. [verificar] SchemaName y EntityName
        # en EntityPartitionSource de la versión de TOM del runtime.
        origen = []
        for p in t.Partitions:
            src = p.Source
            origen.append({
                "tipo": str(p.SourceType),
                "esquema": getattr(src, "SchemaName", None) or None,
                "entidad": getattr(src, "EntityName", None) or None,
                "expresion": getattr(src, "Expression", None) or None,
            })
        tablas.append({
            "nombre": t.Name,
            "descripcion_actual": t.Description or "",
            "oculta": bool(t.IsHidden),
            "modo": ", ".join(modos),
            "origen": origen,
            "columnas": [],
        })

    por_nombre = {t["nombre"]: t for t in tablas}

    # all_columns() ya excluye las columnas internas RowNumber
    for c in tom.all_columns():
        por_nombre[c.Parent.Name]["columnas"].append({
            "nombre": c.Name,
            "tipo_dato": str(c.DataType),
            "tipo": str(c.Type),                      # Data, Calculated, CalculatedTableColumn
            "expresion": getattr(c, "Expression", None) if str(c.Type) == "Calculated" else None,
            "descripcion_actual": c.Description or "",
            "oculta": bool(c.IsHidden),
            "es_clave": bool(c.IsKey),
            "formato": c.FormatString or "",
            "sensible_sugerido": es_sensible(c.Name),
        })

    for m in tom.all_measures():
        medidas.append({
            "tabla": m.Parent.Name,
            "nombre": m.Name,
            "expresion": m.Expression,
            "formato": m.FormatString or "",
            "carpeta": m.DisplayFolder or "",
            "descripcion_actual": m.Description or "",
            "oculta": bool(m.IsHidden),
        })

    for r in tom.model.Relationships:
        relaciones.append({
            "desde_tabla": r.FromTable.Name,
            "desde_columna": r.FromColumn.Name,
            "desde_cardinalidad": str(r.FromCardinality),   # Many / One
            "hacia_tabla": r.ToTable.Name,
            "hacia_columna": r.ToColumn.Name,
            "hacia_cardinalidad": str(r.ToCardinality),
            "activa": bool(r.IsActive),
            "filtro_cruzado": str(r.CrossFilteringBehavior),
        })

print(f"{len(tablas)} tablas, {sum(len(t['columnas']) for t in tablas)} columnas, "
      f"{len(medidas)} medidas, {len(relaciones)} relaciones")

## 2. Perfil de datos: filas, nulos, cardinalidad y muestra

Todo con consultas DAX de lectura (`evaluate_dax`, requiere XMLA de lectura).

In [ ]:
# Cardinalidad, mínimo y máximo de todas las columnas en una sola consulta
try:
    stats = fabric.evaluate_dax(MODELO, "EVALUATE COLUMNSTATISTICS()", workspace=WORKSPACE)
    stats.columns = [re.sub(r"^\[|\]$", "", c) for c in stats.columns]
    stats_idx = {(r["Table Name"], r["Column Name"]): r for _, r in stats.iterrows()}
except Exception as e:  # COLUMNSTATISTICS no está disponible en algunos modos/compatibilidades
    print("COLUMNSTATISTICS no disponible:", e)
    stats_idx = {}

for t in tablas:
    cols = t["columnas"]

    # Filas y nulos: una sola consulta por tabla, columnas por posición
    exprs = [f'"n_filas", COUNTROWS({dax_tabla(t["nombre"])})']
    if CALCULAR_NULOS:
        exprs += [f'"c{i}", COUNTBLANK({dax_columna(t["nombre"], c["nombre"])})' for i, c in enumerate(cols)]
    try:
        fila = fabric.evaluate_dax(MODELO, "EVALUATE ROW(" + ", ".join(exprs) + ")", workspace=WORKSPACE).iloc[0]
        valores = list(fila.values)
        t["filas"] = int(valores[0]) if pd.notna(valores[0]) else 0
        if CALCULAR_NULOS:
            for c, nulos in zip(cols, valores[1:]):
                c["nulos"] = int(nulos) if pd.notna(nulos) else 0
                c["pct_nulos"] = round(100 * c["nulos"] / t["filas"], 2) if t["filas"] else None
    except Exception as e:
        print(f"[{t['nombre']}] sin perfil de filas/nulos:", e)
        t["filas"] = None

    for c in cols:
        s = stats_idx.get((t["nombre"], c["nombre"]))
        if s is not None:
            c["cardinalidad"] = int(s["Cardinality"]) if pd.notna(s["Cardinality"]) else None
            if not c["sensible_sugerido"]:
                c["min"], c["max"] = str(s["Min"]), str(s["Max"])

    # Muestra: TOPN sin orden devuelve N filas cualesquiera; alcanza para ejemplos
    try:
        muestra = fabric.evaluate_dax(
            MODELO, f"EVALUATE TOPN({FILAS_MUESTRA}, {dax_tabla(t['nombre'])})", workspace=WORKSPACE)
        # Las columnas vuelven como Tabla[Columna]
        muestra.columns = [re.sub(r"^.*\[(.*)\]$", r"\1", c) for c in muestra.columns]
        for c in cols:
            if c["sensible_sugerido"]:
                c["ejemplos"] = ["[oculto: sensible]"]
            elif c["nombre"] in muestra.columns:
                c["ejemplos"] = [str(v) for v in muestra[c["nombre"]].dropna().unique()[:3]]
    except Exception as e:
        print(f"[{t['nombre']}] sin muestra:", e)

## 3. Guardar `modelo.json` y `descripciones.csv`

In [ ]:
salida = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", MODELO))
os.makedirs(salida, exist_ok=True)

modelo = {
    "modelo": MODELO,
    "workspace": WORKSPACE,
    "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "tablas": tablas,
    "medidas": medidas,
    "relaciones": relaciones,
}
with open(os.path.join(salida, "modelo.json"), "w", encoding="utf-8") as f:
    json.dump(modelo, f, ensure_ascii=False, indent=2, default=str)

# Plantilla de descripciones: una fila por objeto. Claude (o una persona) completa
# descripcion_nueva y sensibilidad; alguien la revisa y pone revisado = si.
filas = []
for t in tablas:
    filas.append({"tipo_objeto": "tabla", "tabla": t["nombre"], "objeto": t["nombre"],
                  "descripcion_actual": t["descripcion_actual"], "descripcion_nueva": "",
                  "sensibilidad": "", "revisado": "no"})
    for c in t["columnas"]:
        filas.append({"tipo_objeto": "columna", "tabla": t["nombre"], "objeto": c["nombre"],
                      "descripcion_actual": c["descripcion_actual"], "descripcion_nueva": "",
                      "sensibilidad": "Sensible" if c["sensible_sugerido"] else "",
                      "revisado": "no"})
for m in medidas:
    filas.append({"tipo_objeto": "medida", "tabla": m["tabla"], "objeto": m["nombre"],
                  "descripcion_actual": m["descripcion_actual"], "descripcion_nueva": "",
                  "sensibilidad": "", "revisado": "no"})

ruta_csv = os.path.join(salida, "descripciones.csv")
if os.path.exists(ruta_csv):
    print("Ya existe descripciones.csv: no lo piso para no perder descripciones. Borralo a mano si querés regenerarlo.")
else:
    pd.DataFrame(filas).to_csv(ruta_csv, index=False, encoding="utf-8")

sin_desc = sum(1 for f in filas if not f["descripcion_actual"])
print(f"Listo en {salida}")
print(f"{len(filas)} objetos, {sin_desc} sin descripción ({100 * sin_desc // max(len(filas), 1)}%)")